# E-Commerce Supply Chain Analytics

**End-to-end portfolio project | Python • Pandas • Plotly • Business Insights**

## Business objective
Explore a deliberately messy e-commerce supply-chain dataset to understand sales mix, revenue, inventory, supplier/manufacturing lead times, product quality and transportation costs. The goal is to turn raw records into decision-ready insights.

> **Data note:** The bundled dataset is synthetic and generated for practice. Findings demonstrate an analytical workflow; they are not claims about a real company.

### Project workflow
1. Load and profile the raw data
2. Clean inconsistent categories, missing values, invalid values and duplicates
3. Create KPI summaries and explore business questions
4. Build interactive Plotly charts
5. Summarise findings and practical recommendations

## 1. Import libraries and load the raw dataset

Keep `ecommerce_supply_chain_messy_dataset.csv` in the same folder as this notebook. The bundled CSV contains the deliberately messy data used for this analysis.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

pd.set_option('display.max_columns', 30)
px.defaults.template = 'plotly_white'

DATA_PATH = 'ecommerce_supply_chain_messy_dataset.csv'
raw_df = pd.read_csv(DATA_PATH)
df = raw_df.copy()
print(f'Raw dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
display(df.head())

## 2. Initial data-quality audit

Check schema, missing values, duplicate records and suspicious values before changing the data.

In [ ]:
profile = pd.DataFrame({
    'dtype': raw_df.dtypes.astype(str),
    'missing_count': raw_df.isna().sum(),
    'missing_pct': (raw_df.isna().mean() * 100).round(2),
    'unique_values': raw_df.nunique(dropna=True)
}).sort_values('missing_pct', ascending=False)
print('Duplicate rows:', int(raw_df.duplicated().sum()))
display(profile)
display(raw_df.describe(include='all').T)

## 3. Clean and standardise the data

Cleaning choices are explicit and auditable:
- Standardise whitespace and category casing.
- Remove exact duplicate rows.
- Convert numeric fields to numeric types.
- Treat negative prices/stock as invalid; treat defect rates above 20% as implausible outliers for this synthetic dataset.
- Impute missing numeric values with the median and missing categorical values with the mode.

These are project assumptions, not universal rules. In a real business, thresholds and imputation should be confirmed with data owners.

In [ ]:
rows_before = len(df)
duplicates_before = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

# Standardise text columns
text_cols = df.select_dtypes(include='object').columns
for col in text_cols:
    df[col] = df[col].astype('string').str.strip()

# Standardise known categorical values
for col in ['product type', 'availability', 'inspection results', 'transportation modes', 'customer demographics', 'location', 'routes', 'SKU']:
    df[col] = df[col].str.strip()

# Normalise category labels while preserving readable names
for col in ['product type', 'availability', 'inspection results', 'transportation modes']:
    df[col] = df[col].str.lower().str.title()

# Numeric fields
numeric_cols = [
    'price', 'number of product sold', 'revenue generated', 'stock levels',
    'lead times', 'order quantities', 'lead time', 'production volumes',
    'manufacturing lead time', 'manufacturing costs', 'defect rates', 'costs'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Invalid values -> missing, then impute
invalid_price = int((df['price'] < 0).sum())
invalid_stock = int((df['stock levels'] < 0).sum())
invalid_defect = int((df['defect rates'] > 0.20).sum())
df.loc[df['price'] < 0, 'price'] = np.nan
df.loc[df['stock levels'] < 0, 'stock levels'] = np.nan
df.loc[(df['defect rates'] < 0) | (df['defect rates'] > 0.20), 'defect rates'] = np.nan

for col in numeric_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(include=['object', 'string']).columns:
    if df[col].isna().any():
        mode = df[col].mode(dropna=True)
        df[col] = df[col].fillna(mode.iloc[0] if not mode.empty else 'Unknown')

# Business-friendly derived fields
df['estimated_revenue_per_unit'] = df['revenue generated'] / df['number of product sold'].replace(0, np.nan)
df['total_lead_time_days'] = df['lead times'] + df['lead time'] + df['manufacturing lead time']
df['stock_status'] = np.select(
    [df['stock levels'] <= 100, df['stock levels'] <= 400],
    ['Low stock', 'Moderate stock'], default='Healthy stock'
)
df['defect_band'] = pd.cut(
    df['defect rates'], bins=[-0.000001, 0.03, 0.07, 0.20],
    labels=['Low (≤3%)', 'Medium (3–7%)', 'High (>7%)']
).astype(str)

quality_summary = pd.DataFrame({
    'Metric': ['Rows before cleaning', 'Exact duplicate rows removed', 'Invalid negative prices', 'Invalid negative stock values', 'Defect-rate outliers treated as missing', 'Rows after duplicate removal', 'Remaining missing cells'],
    'Value': [rows_before, duplicates_before, invalid_price, invalid_stock, invalid_defect, len(df), int(df.isna().sum().sum())]
})
display(quality_summary)
print('Clean dataset shape:', df.shape)

## 4. KPI overview

The dataset contains product-level observations, not a transaction-level order ledger. Interpret total revenue and units sold as sums across the available records; do not treat row count as the number of orders.

In [ ]:
kpis = pd.Series({
    'Records analysed': len(df),
    'Unique SKUs': df['SKU'].nunique(),
    'Units sold (sum)': int(df['number of product sold'].sum()),
    'Revenue generated (sum)': df['revenue generated'].sum(),
    'Average product price': df['price'].mean(),
    'Average defect rate': df['defect rates'].mean(),
    'Average supplier lead time (days)': df['lead times'].mean(),
    'Average manufacturing lead time (days)': df['manufacturing lead time'].mean(),
    'Transportation cost (sum)': df['costs'].sum()
})
display(kpis.to_frame('Value').style.format({'Value': '{:,.2f}'}))

## 5. Sales and revenue analysis

Questions: Which product categories contribute the most units and revenue? Do high unit sales always correspond to high revenue?

In [ ]:
category_sales = df.groupby('product type', as_index=False).agg(
    units_sold=('number of product sold', 'sum'),
    revenue=('revenue generated', 'sum'),
    avg_price=('price', 'mean'),
    records=('SKU', 'size')
).sort_values('revenue', ascending=False)
display(category_sales.style.format({'revenue':'{:,.0f}', 'avg_price':'{:,.2f}'}))
fig = px.bar(category_sales, x='product type', y='revenue', color='units_sold',
             hover_data=['units_sold','avg_price','records'], title='Revenue by product category',
             labels={'product type':'Product category','revenue':'Revenue generated','units_sold':'Units sold'})
fig.update_layout(xaxis_tickangle=-25)
fig.show()
fig = px.scatter(df, x='number of product sold', y='revenue generated', color='product type',
                 size='price', hover_data=['SKU','price','availability'],
                 title='Units sold vs revenue by record')
fig.show()

## 6. Inventory and availability

Identify low-stock records and compare availability labels. The low-stock threshold is an illustrative rule for this project, not an operational policy.

In [ ]:
inventory_summary = df.groupby(['product type','stock_status'], as_index=False).agg(
    records=('SKU','size'), avg_stock=('stock levels','mean'), units_sold=('number of product sold','sum')
)
display(inventory_summary.sort_values(['product type','records'], ascending=[True,False]))
fig = px.histogram(df, x='stock levels', color='stock_status', nbins=35,
                   title='Distribution of stock levels', marginal='box')
fig.show()
availability_summary = df.groupby('availability', as_index=False).agg(
    records=('SKU','size'), units_sold=('number of product sold','sum'), revenue=('revenue generated','sum')
)
display(availability_summary)
fig = px.bar(availability_summary, x='availability', y='records', color='availability',
             title='Records by availability status')
fig.show()

## 7. Lead-time analysis

Compare supplier, manufacturing and additional lead-time fields. `total_lead_time_days` is a simple sum of the three fields available in this dataset; validate their definitions before using this metric in a real operation.

In [ ]:
lead_summary = df.groupby('product type', as_index=False).agg(
    supplier_lead_days=('lead times','mean'),
    additional_lead_days=('lead time','mean'),
    manufacturing_lead_days=('manufacturing lead time','mean'),
    combined_lead_days=('total_lead_time_days','mean')
).sort_values('combined_lead_days', ascending=False)
display(lead_summary.round(2))
lead_long = lead_summary.melt(id_vars='product type', var_name='lead_time_type', value_name='days')
fig = px.bar(lead_long, x='product type', y='days', color='lead_time_type', barmode='group',
             title='Average lead-time components by product category', labels={'days':'Average days'})
fig.update_layout(xaxis_tickangle=-25)
fig.show()

## 8. Product quality and inspection

Compare average defect rates and inspection outcomes by category. Defect rate is a proportion (for example, 0.05 = 5%).

In [ ]:
quality_by_category = df.groupby('product type', as_index=False).agg(
    avg_defect_rate=('defect rates','mean'),
    inspection_fail_records=('inspection results', lambda s: (s.str.lower() == 'Fail'.lower()).sum()),
    records=('SKU','size')
).sort_values('avg_defect_rate', ascending=False)
display(quality_by_category.style.format({'avg_defect_rate':'{:.2%}'}))
fig = px.bar(quality_by_category, x='product type', y='avg_defect_rate', color='avg_defect_rate',
             title='Average defect rate by product category', labels={'avg_defect_rate':'Average defect rate'})
fig.update_yaxes(tickformat='.0%')
fig.update_layout(xaxis_tickangle=-25)
fig.show()
inspection_summary = df['inspection results'].value_counts().rename_axis('inspection_result').reset_index(name='records')
fig = px.pie(inspection_summary, names='inspection_result', values='records', hole=0.45,
             title='Inspection outcomes')
fig.show()

## 9. Transportation cost analysis

Compare total transportation costs across modes and routes. These are sums of record-level costs, so use the same record scope when comparing groups.

In [ ]:
transport_summary = df.groupby('transportation modes', as_index=False).agg(
    total_transport_cost=('costs','sum'),
    avg_transport_cost=('costs','mean'),
    records=('SKU','size')
).sort_values('total_transport_cost', ascending=False)
display(transport_summary.style.format({'total_transport_cost':'{:,.2f}','avg_transport_cost':'{:,.2f}'}))
fig = px.bar(transport_summary, x='transportation modes', y='total_transport_cost', color='transportation modes',
             title='Total transportation cost by mode')
fig.show()
route_summary = df.groupby('routes', as_index=False).agg(
    total_cost=('costs','sum'), avg_cost=('costs','mean'), records=('SKU','size')
).sort_values('total_cost', ascending=False)
display(route_summary.round(2))
fig = px.bar(route_summary, x='routes', y='avg_cost', color='total_cost',
             title='Average transportation cost by route', hover_data=['total_cost','records'])
fig.show()

## 10. Interactive exploration

Use the dropdowns to focus on a product category, location and transportation mode. Charts and summary table update to the filtered data.

In [ ]:
# Interactive Plotly chart: hover to inspect each category, and use the legend to show/hide categories.
interactive_view = df[df['location'].isin(df['location'].value_counts().head(5).index)].copy()
interactive_summary = interactive_view.groupby(['location', 'product type'], as_index=False).agg(
    revenue=('revenue generated', 'sum'), units_sold=('number of product sold', 'sum')
)
fig = px.bar(interactive_summary, x='location', y='revenue', color='product type',
             hover_data=['units_sold'], barmode='stack',
             title='Interactive revenue view: five locations with most records')
fig.show()

## 11. Automated insight summary

This section calculates findings from the cleaned data rather than hard-coding a winner.

In [ ]:
best_revenue = category_sales.loc[category_sales['revenue'].idxmax()]
best_units = category_sales.loc[category_sales['units_sold'].idxmax()]
highest_defect = quality_by_category.loc[quality_by_category['avg_defect_rate'].idxmax()]
most_costly_mode = transport_summary.loc[transport_summary['total_transport_cost'].idxmax()]
lowest_stock_category = df.groupby('product type')['stock levels'].mean().idxmin()
print(f"• Highest revenue category: {best_revenue['product type']} ({best_revenue['revenue']:,.2f}).")
print(f"• Highest unit-sales category: {best_units['product type']} ({best_units['units_sold']:,.0f} units summed across records).")
print(f"• Highest average defect-rate category: {highest_defect['product type']} ({highest_defect['avg_defect_rate']:.2%}).")
print(f"• Highest total transportation-cost mode: {most_costly_mode['transportation modes']} ({most_costly_mode['total_transport_cost']:,.2f}).")
print(f"• Lowest average stock category: {lowest_stock_category}.")

## 12. Recommendations and limitations

### Possible actions to investigate
- Review categories with higher defect rates and inspection failures; validate product-level patterns before changing supplier contracts.
- Investigate low-stock categories alongside sales volume before setting replenishment thresholds.
- Compare average transport cost per record and route mix, not just total cost, before changing transportation modes.
- Review long lead-time components with suppliers and manufacturing teams.
- In production, replace the synthetic dataset with validated order, inventory, quality and logistics tables, and define the grain of each table.

### Limitations
- Synthetic, randomly generated records; patterns are for portfolio demonstration only.
- No real dates or order IDs, so this dataset cannot support time-series trends, on-time delivery rate, order-level profitability or true customer retention analysis.
- SKU labels may repeat across generated rows and are not guaranteed to represent a unique product master record.
- Missing-value imputation and outlier thresholds are assumptions that should be reviewed with business stakeholders.

## 13. Export cleaned data

In [ ]:
clean_path = 'ecommerce_supply_chain_cleaned.csv'
df.to_csv(clean_path, index=False)
print(f'Cleaned data exported to: {clean_path}')
print(f'Final data quality check — missing cells: {int(df.isna().sum().sum())}, duplicate rows: {int(df.duplicated().sum())}')